# 08 Stanford Foundation Model Transparency Index (December 2025)

**Purpose.** This notebook reads the Stanford CRFM Foundation Model Transparency Index (FMTI), December 2025 edition, and turns it into three tidy tables: one row per developer and indicator, one row per developer with total and domain scores, and one row per indicator with its definition. The index scores 13 developers on 100 yes or no transparency indicators about data, models and downstream use. In the project it gives context for Q5 (what each developer discloses) next to the data-handling terms we collected in Search A.

**Question served.** Q5 background. It is a reserve source and does not replace Search A.

**How to run.** Run all cells from top to bottom. The notebook tries to download the repository archive from GitHub. If that is blocked, it uses the file `fmti-main.zip` that you saved by hand in `Data/raw/manual/`.

## At a glance: links, access, download, timing

| Item | Detail |
|---|---|
| Kind | Download of one ZIP (the GitHub archive of the repository), read in memory |
| Publisher | Stanford Center for Research on Foundation Models (CRFM), with Princeton and MIT researchers |
| Results page | https://crfm.stanford.edu/fmti/December-2025 |
| Paper | https://crfm.stanford.edu/fmti/December-2025/paper.pdf |
| Data repository | https://github.com/stanford-crfm/fmti |
| File used | https://github.com/stanford-crfm/fmti/archive/refs/heads/main.zip (saved as `fmti-main.zip`) |
| Access | Public repository. No account, key or login needed. No scraping |
| Download size | about 15 MB for the whole archive. Only three kinds of CSV inside are read |
| Expected time | download 2 to 15 seconds, whole notebook under 30 seconds |
| Data as of | December 2025 edition. See `processed/data_currency.csv` |
| To update | Run the notebook again. A newer edition would sit in a new folder of the repository, see the update section |

## 1. Source and reference links

| Item | Link |
|---|---|
| FMTI project page | https://crfm.stanford.edu/fmti/ |
| December 2025 results page | https://crfm.stanford.edu/fmti/December-2025 |
| December 2025 paper (PDF) | https://crfm.stanford.edu/fmti/December-2025/paper.pdf |
| Company transparency reports | https://crfm.stanford.edu/fmti/December-2025/company-reports/index.html |
| Data repository | https://github.com/stanford-crfm/fmti |
| Archive download | https://github.com/stanford-crfm/fmti/archive/refs/heads/main.zip |
| October 2023 paper (first edition) | https://arxiv.org/abs/2310.12941 |

**Citation.** Bommasani, Klyman, Kapoor, Longpre, Xiong, Maslej and Liang, Foundation Model Transparency Index, December 2025 edition, Stanford CRFM. Use the citation block in the repository README.

**Scope note.** The index scores the company, based on the disclosures the company made about its flagship model. A score of 1 means the indicator was satisfied on the basis of public information or the company's own report. It does not say whether the company's practice is good, only whether it is disclosed.

## 2. Table structure

### 2.1 Structure of the source files (all inside `fmti-main/Dec2025/`)

| File | Content |
|---|---|
| `Dec2025_scores.csv` | 100 rows (one per indicator) and 13 developer columns. Each cell is 1 (satisfied) or 0 (not satisfied) |
| `Dec2025_indicators.csv` | 100 rows. Columns `Domain` (Upstream, Model or Downstream), `Subdomain`, `Indicator`, `Definition`, `Notes`, `Example disclosure` |
| `reports/<Developer>_fmtransparencyreport_Dec2025.csv` | One file per developer. Columns `Indicator`, `Definition`, `Notes`, `Disclosure`, `Justification`, `References`, `Score` |

### 2.2 Structure of the outputs in `processed/`

| File | Rows | Columns |
|---|---|---|
| `fmti_indicator_scores.csv` | 1,300 (13 developers times 100 indicators) | `developer`, `indicator`, `domain`, `subdomain`, `score` (0 or 1), `reference_count` (number of web addresses cited in the developer's report for this indicator) |
| `fmti_company_scores.csv` | 13 | `developer`, `total_score` (out of 100), `total_pct`, `upstream_score`, `model_score`, `downstream_score`, `rank`, `hq_country`, `is_china_based` (our annotation, see Section 5) |
| `fmti_indicators.csv` | 100 | `domain`, `subdomain`, `indicator`, `definition` |
| `data_currency.csv` | 1 | when extracted, what period the data covers |

## 3. Set up the paths and download helper

In [1]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

T0 = time.time()
FOLDER = "08_fmti_transparency"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
elif (cwd / "Data" / FOLDER).exists():
    HERE = cwd / "Data" / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER} (or from Data/). Current folder: {cwd}")

DATA = HERE.parent               # the Data folder
MANUAL = DATA / "raw" / "manual"                # hand-downloaded or pre-copied source files
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)


HEADERS = {"User-Agent": "UTS-36104-A3-student-project (academic use, no contact data stored)"}
TIMEOUT = 120
REFRESH = True   # True: try a fresh download first. False: reuse the file already in raw/ if there is one.


def record(source_id: str, url: str, path: Path, how: str) -> None:
    """Add (or replace) the retrieval-log entry for one raw file."""
    content = path.read_bytes()
    log = json.loads(LOG_PATH.read_text(encoding="utf-8")) if LOG_PATH.exists() else {}
    log[source_id] = {
        "url": url,
        "file": path.name,
        "how": how,
        "retrieved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")
    print(f"logged {source_id}: {len(content):,} bytes via {how}")


def get_file(source_id: str, url: str, target: str, manual_name: str = None) -> Path:
    """Return the path of a raw file. Order: fresh download, earlier download in raw/, manual copy."""
    path = RAW / target
    if REFRESH or not path.exists():
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            path.write_bytes(response.content)
            record(source_id, url, path, "downloaded")
            return path
        except Exception as err:
            print(f"Download of {source_id} failed: {err}")
    manual = MANUAL / (manual_name or target)
    candidates = [p for p in (path, manual) if p.exists()]
    if candidates:
        best = max(candidates, key=lambda p: p.stat().st_mtime)   # the newest copy wins
        if best == path:
            print(f"Using the earlier download {path.name}")
        else:
            shutil.copy2(best, path)
            record(source_id, url, path, f"copied from manual folder {manual.parent.name}")
        return path
    raise FileNotFoundError(
        f"Could not get {source_id}. Open {url} in a browser, save the file as '{manual_name or target}' "
        f"inside {MANUAL}, then run this cell again.")

import io
import zipfile
import re

Notebook folder: d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\08_fmti_transparency
Manual folder  : d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\raw\manual


## 4. Acquire the archive

The notebook first tries the GitHub archive. If that fails, it uses `fmti-main.zip` from `Data/raw/manual/`. A copy lying loose in the `Data` folder is also accepted and is moved into `Data/raw/manual/` by copying.

In [2]:
FMTI_URL = "https://github.com/stanford-crfm/fmti/archive/refs/heads/main.zip"
MANUAL.mkdir(parents=True, exist_ok=True)
loose = DATA / "fmti-main.zip"
if loose.exists() and not (MANUAL / "fmti-main.zip").exists():
    shutil.copy2(loose, MANUAL / "fmti-main.zip")
    print("Copied the loose fmti-main.zip into", MANUAL)

zip_path = get_file("fmti_repository", FMTI_URL, "fmti-main.zip")
archive = zipfile.ZipFile(zip_path)
names = archive.namelist()
print(len(names), "entries in the archive")

def find(suffix):
    hits = [x for x in names if x.endswith(suffix)]
    assert hits, f"{suffix} not found in the archive. The repository layout may have changed."
    return hits[0]

SCORES_FILE = find("Dec2025/Dec2025_scores.csv")
INDICATORS_FILE = find("Dec2025/Dec2025_indicators.csv")
REPORT_FILES = sorted(x for x in names if "/Dec2025/reports/" in x and x.endswith(".csv"))
print("scores    :", SCORES_FILE)
print("indicators:", INDICATORS_FILE)
print("reports   :", len(REPORT_FILES))

logged fmti_repository: 15,145,526 bytes via downloaded
108 entries in the archive
scores    : fmti-main/Dec2025/Dec2025_scores.csv
indicators: fmti-main/Dec2025/Dec2025_indicators.csv
reports   : 13


## 5. Read the three kinds of file

In [3]:
def read_csv(member):
    return pd.read_csv(io.BytesIO(archive.read(member)))

scores_wide = read_csv(SCORES_FILE)
indicators = read_csv(INDICATORS_FILE)
for col in ("Domain", "Subdomain", "Indicator", "Definition"):
    indicators[col] = indicators[col].astype(str).str.strip()
scores_wide["Indicator"] = scores_wide["Indicator"].astype(str).str.strip()

print("scores     :", scores_wide.shape)
print("indicators :", indicators.shape)
print("developers :", list(scores_wide.columns[1:]))
print()
print(indicators.groupby("Domain").size())

scores     : (100, 14)
indicators : (100, 6)
developers : ['AI21 Labs', 'Alibaba', 'Amazon', 'Anthropic', 'DeepSeek', 'Google', 'IBM', 'Meta', 'Midjourney', 'Mistral', 'OpenAI', 'Writer', 'xAI']

Domain
Downstream    36
Model         30
Upstream      34
dtype: int64


In [4]:
# Count the web addresses each developer cites for each indicator.
URL = re.compile(r"https?://[^\s]+")
ref_rows = []
for member in REPORT_FILES:
    rep = read_csv(member)
    developer = Path(member).name.split("_fmtransparencyreport")[0].replace("_", " ")
    rep["Indicator"] = rep["Indicator"].astype(str).str.strip()
    for _, r in rep.iterrows():
        ref_rows.append({"developer": developer, "indicator": r["Indicator"],
                         "report_score": r["Score"],
                         "reference_count": len(URL.findall(str(r["References"])) ) if pd.notna(r["References"]) else 0})
refs = pd.DataFrame(ref_rows)
print(refs.shape, "rows from", refs["developer"].nunique(), "developer reports")

(1300, 4) rows from 13 developer reports


## 6. Build the tidy tables

The developer's headquarters country is **our annotation**, typed in by hand from public knowledge of each company (it is not part of the FMTI data). It is used only to flag the two China-based developers the index scored for the first time.

In [5]:
HQ_COUNTRY = {   # our annotation, headquarters country of the developer
    "AI21 Labs": "Israel", "Alibaba": "China", "Amazon": "United States", "Anthropic": "United States",
    "DeepSeek": "China", "Google": "United States", "IBM": "United States", "Meta": "United States",
    "Midjourney": "United States", "Mistral": "France", "OpenAI": "United States", "Writer": "United States",
    "xAI": "United States",
}

meta = indicators.rename(columns={"Domain": "domain", "Subdomain": "subdomain", "Indicator": "indicator",
                                  "Definition": "definition"})[["domain", "subdomain", "indicator", "definition"]]

long = scores_wide.melt(id_vars="Indicator", var_name="developer", value_name="score").rename(columns={"Indicator": "indicator"})
long = long.merge(meta[["indicator", "domain", "subdomain"]], on="indicator", how="left")
long = long.merge(refs[["developer", "indicator", "reference_count"]], on=["developer", "indicator"], how="left")
long = long[["developer", "indicator", "domain", "subdomain", "score", "reference_count"]]

totals = long.pivot_table(index="developer", columns="domain", values="score", aggfunc="sum").fillna(0).astype(int)
totals.columns = [f"{c.lower()}_score" for c in totals.columns]
company = totals.assign(total_score=long.groupby("developer")["score"].sum()).reset_index()
company["total_pct"] = (company["total_score"] / long["indicator"].nunique() * 100).round(1)
company["rank"] = company["total_score"].rank(ascending=False, method="min").astype(int)
company["hq_country"] = company["developer"].map(HQ_COUNTRY)
company["is_china_based"] = company["hq_country"].eq("China")
company = company[["developer", "total_score", "total_pct", "upstream_score", "model_score", "downstream_score",
                   "rank", "hq_country", "is_china_based"]].sort_values("rank").reset_index(drop=True)
company

,developer,total_score,total_pct,upstream_score,model_score,downstream_score,rank,hq_country,is_china_based
0,IBM,95,95.0,34,28,33,1,United States,False
1,Writer,72,72.0,24,18,30,2,United States,False
2,AI21 Labs,66,66.0,18,21,27,3,Israel,False
3,Anthropic,46,46.0,3,17,26,4,United States,False
4,Google,41,41.0,8,13,20,5,United States,False
5,Amazon,39,39.0,6,15,18,6,United States,False
6,OpenAI,35,35.0,1,13,21,7,United States,False
7,DeepSeek,32,32.0,11,14,7,8,China,True
8,Meta,31,31.0,7,12,12,9,United States,False
9,Alibaba,26,26.0,6,12,8,10,China,True


## 7. Validate

In [6]:
problems = []
if long.shape[0] != 1300:
    problems.append(f"expected 1,300 developer-indicator rows, found {long.shape[0]}")
if not set(long["score"].unique()) <= {0, 1}:
    problems.append("scores other than 0 or 1")
if long["domain"].isna().any():
    problems.append("indicators without a domain (names in the scores file and the indicator file differ)")
if indicators["Indicator"].duplicated().any():
    problems.append("duplicated indicator names")
# The score in each developer's own report must equal the score in the wide table.
check = long.merge(refs[["developer", "indicator", "report_score"]], on=["developer", "indicator"], how="left")
mismatch = int((check["score"] != check["report_score"]).sum())
if mismatch:
    problems.append(f"{mismatch} rows where the report score differs from the scores table")
if company["developer"].isna().any() or company["hq_country"].isna().any():
    problems.append("developer without a headquarters annotation. Add it to HQ_COUNTRY")
print("Problems:", problems if problems else "none")
assert not problems, problems
print()
print("Developers:", company.shape[0], " Indicators:", meta.shape[0])
print("Highest:", company.iloc[0]["developer"], company.iloc[0]["total_score"], " Lowest:", company.iloc[-1]["developer"], company.iloc[-1]["total_score"])

Problems: none

Developers: 13  Indicators: 100
Highest: IBM 95  Lowest: xAI 14


## 8. Save

In [7]:
long.to_csv(OUT / "fmti_indicator_scores.csv", index=False, encoding="utf-8")
company.to_csv(OUT / "fmti_company_scores.csv", index=False, encoding="utf-8")
meta.to_csv(OUT / "fmti_indicators.csv", index=False, encoding="utf-8")
print("saved 3 tables:", long.shape, company.shape, meta.shape)

saved 3 tables: (1300, 6) (13, 9) (100, 4)


## 9. Quick look: China-based developers against the rest

Average score per domain. The index has only two China-based developers (Alibaba and DeepSeek), so this is a comparison of two against eleven and should be read as an illustration, not a pattern.

In [8]:
cols = ["total_score", "upstream_score", "model_score", "downstream_score"]
company.groupby("is_china_based")[cols].agg(["mean", "count"]).round(1)

total_score       upstream_score       model_score        \
                      mean count           mean count        mean count   
is_china_based                                                            
False                 42.8    11            9.3    11        13.9    11   
True                  29.0     2            8.5     2        13.0     2   

               downstream_score        
                           mean count  
is_china_based                         
False                      19.6    11  
True                        7.5     2

## How to update, and what to do if the download is blocked

**Automatic.** Run all cells again. Section 4 downloads the current main branch of the repository and replaces `raw/fmti-main.zip`. About 10 seconds.

**Manual download (use this if you see `Download of fmti_repository failed`).**

1. Open https://github.com/stanford-crfm/fmti in a browser.
2. Click the green **Code** button, then **Download ZIP**. Or paste https://github.com/stanford-crfm/fmti/archive/refs/heads/main.zip into the address bar.
3. Save it as `Data\raw\manual\fmti-main.zip`, replacing the old copy. Do not unpack it.
4. Run the notebook again. With no fresh download, it uses the **newer** of `raw/fmti-main.zip` and the manual copy.

**When a newer edition appears.** The repository keeps one folder per edition (`Oct2023`, `May2024`, `Dec2025`). If Stanford adds a newer folder, change `Dec2025` in Section 4 and in the file names of Section 2 to the new edition label, and check that the 100 indicators and 13 developers counts in Section 7 still hold. The validation will stop the run with a message if the layout changed.

**Update check.** Open `raw/retrieval_log.json` and read the `how` field (`downloaded` or `copied from manual folder`). Then open `processed/data_currency.csv`.

## 10. Run time and limits

**Time to extract.** One request for an archive of about 15 MB, then reading 15 small CSV files in memory. Expect 2 to 15 seconds for the download and under 3 seconds for the reading, so a full run takes **under 30 seconds**. The last cell prints the measured time.

**Limits and known risks.**

1. The index scores companies, not individual models. Joining it to Epoch, LiteLLM or LMArena rows needs a developer-to-model step that is not built here.
2. Only 13 developers are covered, and 2 of them are China-based. Z.ai (Zhipu) and Moonshot, which Search A covers, are not in the index.
3. A score of 0 means the indicator was not satisfied on public information. It does not prove the practice does not exist.
4. Headquarters country is our annotation and is not part of the Stanford data.
5. The scores are one snapshot (December 2025). Companies may have changed their disclosures since.

## 11. References

1. Stanford CRFM, FMTI December 2025. https://crfm.stanford.edu/fmti/December-2025
2. Stanford CRFM, FMTI December 2025 paper. https://crfm.stanford.edu/fmti/December-2025/paper.pdf
3. Stanford CRFM, data repository. https://github.com/stanford-crfm/fmti
4. Bommasani et al., first edition paper, October 2023. https://arxiv.org/abs/2310.12941

## Data currency: how up to date is this data?

Every notebook writes `processed/data_currency.csv` with the day it was extracted, the dates the data itself covers, and what we know about when the source started and how often it changes. Use these values for the "data as of" label in the app.

In [9]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    """Record how current the data is. extracted_utc is the moment this notebook ran."""
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": str(covers_from)[:10] if covers_from is not None else "not stated in the data",
        "data_covers_to": str(covers_to)[:10] if covers_to is not None else "not stated in the data",
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


write_currency("Stanford FMTI December 2025 (fmti_company_scores.csv)",
               "2025-12", "2025-12",
               "One edition of the index (December 2025). Scores are fixed for that edition and do not change when the repository is downloaded again",
               "First edition October 2023, second May 2024, third December 2025 (13 developers). The repository README gives the history",
               "Irregular, roughly one edition per year")

dataset                  Stanford FMTI December 2025 (fmti_company_scores.csv)
extracted_utc            2026-10-17T22:51:37Z
data_covers_from         2025-12
data_covers_to           2025-12
snapshot_note            One edition of the index (December 2025). Scores are fixed for that edition and do not change when the repository is downloaded again
source_origin_note       First edition October 2023, second May 2024, third December 2025 (13 developers). The repository README gives the history
source_update_frequency  Irregular, roughly one edition per year


In [10]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")

Elapsed time of this run: 4.9 seconds
Files written:
  processed\data_currency.csv  (504 bytes)
  processed\fmti_company_scores.csv  (681 bytes)
  processed\fmti_indicator_scores.csv  (80,506 bytes)
  processed\fmti_indicators.csv  (13,768 bytes)
